<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 09 — K-Means e Aprendizado Não Supervisionado
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Descobrindo Grupos Ocultos nos Passageiros do Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🔍 Não Supervisionado</span>
</div>


## Uma mudança de paradigma

Ao longo de todo o curso usamos dados **rotulados**: sabíamos que cada
passageiro sobreviveu (1) ou não (0), e nossos modelos aprenderam a prever
esse rótulo. Isso é o **aprendizado supervisionado** — temos um "gabarito"
durante o treino.

Hoje mudamos de paradigma:

> *"E se fingíssemos que não sabemos quem sobreviveu? Será que o algoritmo*
> *consegue descobrir grupos naturais nos dados — sozinho?"*

Esse é o **aprendizado não supervisionado**: sem rótulos, sem gabarito.
O algoritmo descobre a estrutura por conta própria.

### O experimento de hoje

1. **Escondemos a coluna `survived`** — o algoritmo não pode vê-la
2. Aplicamos o **K-Means** para encontrar grupos de passageiros similares
3. Depois de formados os grupos, **revelamos os rótulos reais**
4. Analisamos: os grupos correspondem a padrões reais de sobrevivência?

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Supervisionado vs Não Supervisionado |
| **2** | Intuição do K-Means — 4 passos que se repetem |
| **3** | Escolhendo K — Cotovelo e Silhouette Score |
| **4** | Aplicando K-Means no Titanic |
| **5** | A revelação — o que o algoritmo encontrou? |

> **Tempo estimado: 35 minutos**

<div style="background:#d1ecf1; border-left:5px solid #0c5460; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#0c5460;">Aplicações reais do K-Means:</strong>
<span style="color:#0c5460;"> segmentação de clientes em e-commerce, agrupamento de notícias por tema, compressão de imagens, detecção de anomalias. É um dos algoritmos mais usados na indústria.</span>
</div>


In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() or 4)   # evita um warning cosmético do KMeans no Windows

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.preprocessing import StandardScaler

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# Features usadas para o CLUSTERING — sem 'survived'!
FEATURES_CLUSTER = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X_cluster = df[FEATURES_CLUSTER].copy()

# Normalização (K-Means é baseado em distância — obrigatória!)
scaler = StandardScaler()
X_sc   = scaler.fit_transform(X_cluster)

# Guardando os rótulos reais — só serão revelados na Parte 5
y_real = df["survived"].values

print("✅ Dataset pronto para clustering!")
print(f"   {len(df)} passageiros | {len(FEATURES_CLUSTER)} features")
print()
print("IMPORTANTE: a coluna 'survived' foi OCULTADA do algoritmo.")
print("O K-Means não sabe quem sobreviveu — vai descobrir sozinho os grupos.")
print(f"Features usadas: {FEATURES_CLUSTER}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Supervisionado vs Não Supervisionado</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"A diferença está em quem conhece a resposta antes de começar."</p>
    </div>
</div>

### Os dois paradigmas

```
SUPERVISIONADO                       NÃO SUPERVISIONADO
─────────────────────────────────    ─────────────────────────────────
Dados:  (X, Y) — com rótulos         Dados: (X) — sem rótulos
Tarefa: aprender X → Y               Tarefa: descobrir estrutura em X
Saída:  previsão para novos X        Saída: grupos, padrões
Avalia: comparando ŷ com Y real      Avalia: coesão e separação dos grupos
```

### Onde o K-Means se encaixa?

O **K-Means** é o algoritmo de **clustering** mais utilizado. Ele divide os
dados em **K grupos** de forma que pontos do mesmo grupo sejam o mais
similares possível entre si, e pontos de grupos diferentes sejam o mais
diferentes possível.

O gráfico abaixo mostra os passageiros usando apenas duas características
(idade e tarifa) — para você ver a diferença entre trabalhar **com** e **sem**
os rótulos de sobrevivência. O K-Means só enxerga o gráfico da esquerda.


In [ ]:
# Visualizando a diferença: com e sem rótulos (usando idade x tarifa)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Supervisionado vs Não Supervisionado — Titanic", fontweight="bold")

axes[0].scatter(df["age"], df["fare"], color="#0f3460", s=15, alpha=0.4, edgecolors="none")
axes[0].set_title("Situação do K-Means\n(sem rótulos — só vemos os pontos)", fontweight="bold")
axes[0].set_xlabel("Idade"); axes[0].set_ylabel("Tarifa (£)")

cores_surv = ["#e94560" if s == 0 else "#0f3460" for s in y_real]
axes[1].scatter(df["age"], df["fare"], c=cores_surv, s=15, alpha=0.5, edgecolors="none")
axes[1].set_title("Com rótulos reais revelados\n(informação que o K-Means não tem acesso)",
                  fontweight="bold")
axes[1].set_xlabel("Idade"); axes[1].set_ylabel("Tarifa (£)")

import matplotlib.patches as mpatches
axes[1].legend(handles=[mpatches.Patch(color="#0f3460", label="Sobreviveu"),
                         mpatches.Patch(color="#e94560", label="Não Sobreviveu")])

plt.tight_layout()
plt.show()

print("Desafio do algoritmo: olhando apenas o gráfico da esquerda,")
print("conseguir descobrir os grupos naturais que vemos na direita?")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Olhando o gráfico da esquerda (sem rótulos): (a) você consegue identificar visualmente grupos naturais? (b) os dados parecem bem separados ou muito misturados?</span></div>

*✏️ (a) Grupos visíveis: `???`*

*✏️ (b) Dados estão `???` separados*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição do K-Means — O Algoritmo Passo a Passo</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Quatro passos simples que se repetem até convergir."</p>
    </div>
</div>

### O algoritmo em 4 passos

```
1. INICIALIZAÇÃO
   Escolher K centróides aleatoriamente (scikit-learn usa K-Means++)

2. ATRIBUIÇÃO
   Para cada ponto: calcular distância a todos os centróides
   Atribuir o ponto ao centróide mais próximo

3. ATUALIZAÇÃO
   Para cada cluster: recalcular o centróide
   como a MÉDIA de todos os pontos daquele cluster

4. REPETIR os passos 2 e 3 até os centróides não se moverem mais
```

### O que o algoritmo minimiza?

O K-Means minimiza a **WCSS** (soma das distâncias ao quadrado de cada ponto
ao centróide do seu cluster). Quanto menor o WCSS, mais **compactos** são os
clusters.

<div style="background:#fff3cd; border-left:5px solid #856404; padding:12px 18px; border-radius:6px; margin:12px 0;">
<strong style="color:#856404;">K-Means pode convergir para soluções ruins</strong> dependendo da inicialização aleatória.
<span style="color:#856404;"> O scikit-learn usa <strong>K-Means++</strong> por padrão e repete o processo várias vezes (<code>n_init=10</code>), mantendo o melhor resultado.</span>
</div>


In [ ]:
# Implementando K-Means do zero — para ver a convergência acontecendo
np.random.seed(42)
from sklearn.datasets import make_blobs

X_demo, _ = make_blobs(n_samples=120, centers=3, cluster_std=0.8, random_state=42)

def kmeans_manual(X, K, n_iter=6, seed=10):
    np.random.seed(seed)
    idx = np.random.choice(len(X), K, replace=False)
    centroides = X[idx].copy()
    historico = [centroides.copy()]
    labels_hist = []
    for _ in range(n_iter):
        dists  = np.array([[np.linalg.norm(x - c) for c in centroides] for x in X])
        labels = np.argmin(dists, axis=1)
        labels_hist.append(labels.copy())
        centroides = np.array([X[labels == k].mean(axis=0) for k in range(K)])
        historico.append(centroides.copy())
    return historico, labels_hist

K = 3
historico, labels_hist = kmeans_manual(X_demo, K)

CORES = ["#0f3460", "#e94560", "#f0a500"]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("K-Means — Convergência Iteração por Iteração (K=3)", fontsize=13, fontweight="bold")

for ax, it in zip(axes, [0, 1, 2, 5]):
    labels_it = labels_hist[min(it, len(labels_hist)-1)]
    cent_it   = historico[it]
    for k in range(K):
        mask = labels_it == k
        ax.scatter(X_demo[mask,0], X_demo[mask,1], color=CORES[k], s=30, alpha=0.6, edgecolors="none")
    for k in range(K):
        ax.scatter(*cent_it[k], color=CORES[k], s=300, marker="*",
                   edgecolors="white", linewidth=1.5, zorder=10)
    if it > 0:
        cent_ant = historico[it-1]
        for k in range(K):
            ax.annotate("", xy=cent_it[k], xytext=cent_ant[k],
                        arrowprops=dict(arrowstyle="->", color="#333", lw=1.5, alpha=0.7))
    ax.set_title("Inicialização" if it == 0 else f"Iteração {it}", fontweight="bold")
    ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

print("Observe: os centróides (estrelas) se movem a cada iteração")
print("e param quando chegam ao centro de cada grupo.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Analise as 4 imagens: (a) na inicialização, os centróides já estão nos centros dos grupos? (b) após quantas iterações eles parecem ter convergido?</span></div>

*✏️ (a) Na inicialização: `???`*

*✏️ (b) Convergência após ~`???` iterações*


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# print("(a) Na inicialização aleatória, os centróides raramente estão nos")
# print("    centros reais. A convergência parte de longe do ótimo, mas chega")
# print("    lá em poucas iterações.")
# print()
# print("(b) Para 3 grupos bem separados, o K-Means converge em 3-6 iterações.")
# print("    Para grupos sobrepostos, pode levar mais tempo ou não convergir bem.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Escolhendo K — Cotovelo e Silhouette Score</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O maior desafio do K-Means: quantos grupos existem nos dados?"</p>
    </div>
</div>

### Método do Cotovelo

O K-Means exige que você defina quantos grupos existem. Uma forma de decidir:
treinar para K = 1, 2, 3... e plotar o WCSS. O **"cotovelo"** é o ponto onde a
redução do WCSS começa a diminuir — depois dele, mais clusters só fragmentam
os dados sem ganho real.

### Silhouette Score — confirmando a escolha

O WCSS só mede o quão compactos são os clusters, não se estão bem
**separados** entre si. O Silhouette Score mede as duas coisas:

```
         b(i) − a(i)
s(i) = ────────────────
          max(a(i), b(i))

  a(i) = distância média aos pontos do MESMO cluster (coesão)
  b(i) = distância média aos pontos do cluster vizinho mais próximo (separação)
```

| Valor | Significado |
|-------|-------------|
| **≈ +1** | Ponto bem dentro do seu cluster, longe dos outros |
| **≈ 0** | Ponto na fronteira entre dois clusters |
| **≈ −1** | Ponto provavelmente no cluster errado |


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Calculando WCSS e Silhouette para K de 2 a 8
Ks    = range(2, 9)
wcss  = []
sils  = []

for k in Ks:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels_k = km.fit_predict(X_sc)
    wcss.append(km.inertia_)
    sils.append(silhouette_score(X_sc, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Escolhendo K — Cotovelo e Silhouette", fontweight="bold")

axes[0].plot(Ks, wcss, "o-", color="#0f3460", linewidth=2.5, markersize=8)
axes[0].set_xlabel("K"); axes[0].set_ylabel("WCSS (Inércia)")
axes[0].set_title("Método do Cotovelo", fontweight="bold")

axes[1].plot(Ks, sils, "s-", color="#e94560", linewidth=2.5, markersize=8)
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score por K", fontweight="bold")

plt.tight_layout()
plt.show()

melhor_K_sil = list(Ks)[np.argmax(sils)]
print(f"K com maior Silhouette Score: {melhor_K_sil} ({max(sils):.4f})")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Analise os dois gráficos: (a) em qual K você vê o "cotovelo" no primeiro gráfico? (b) esse K coincide com o de maior Silhouette Score? (c) por que não escolher K=8 mesmo com WCSS menor?</span></div>

*✏️ (a) Cotovelo em K = `???`*

*✏️ (b) Coincide com o Silhouette? `???`*

*✏️ (c) Não escolhemos K=8 porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# print("(a)/(b) O cotovelo geralmente cai entre K=2 e K=4 no Titanic — muitas vezes")
# print("    coincide (ou fica perto) do K de maior Silhouette Score.")
# print()
# print("(c) O WCSS sempre cai com mais clusters (até K=n, cada ponto é um cluster).")
# print("    Clusters demais são difíceis de interpretar e perdem significado.")
# print("    O objetivo é encontrar grupos ÚTEIS, não minimizar WCSS a qualquer custo.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Aplicando K-Means no Titanic</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Com K definido, vamos treinar e analisar o que o algoritmo encontrou."</p>
    </div>
</div>

Vamos usar **K=3**, um valor razoável tanto pelo cotovelo quanto pelo
Silhouette, e que também faz sentido pelo conhecimento do domínio (o Titanic
tem grupos naturais de classe social e gênero).


In [ ]:
K_escolhido = 3
km_final = KMeans(n_clusters=K_escolhido, init="k-means++", n_init=10, random_state=42)
labels_final = km_final.fit_predict(X_sc)

df_analise = df.copy()
df_analise["cluster"] = labels_final

print(f"K-Means treinado com K={K_escolhido}")
print(f"Silhouette Score: {silhouette_score(X_sc, labels_final):.4f}")
print()
for c in range(K_escolhido):
    n = (labels_final == c).sum()
    print(f"  Cluster {c}: {n} passageiros ({n/len(df):.0%})")


In [ ]:
# Visualizando os clusters no espaço idade x tarifa
plt.figure(figsize=(8, 6))
cmap_k = cm.get_cmap("tab10")

for c in range(K_escolhido):
    mask = labels_final == c
    plt.scatter(df["age"][mask], df["fare"][mask], color=cmap_k(c/K_escolhido),
                s=20, alpha=0.6, edgecolors="none", label=f"Cluster {c} (n={mask.sum()})")

plt.xlabel("Idade"); plt.ylabel("Tarifa (£)")
plt.title(f"Clusters do K-Means (K={K_escolhido}) — vistos por Idade x Tarifa", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

print("Nota: o K-Means usou 5 features para formar os grupos (não só idade e")
print("tarifa) — este gráfico é apenas uma janela 2D para visualizar o resultado.")


### Perfil de cada cluster

Antes de revelar `survived`, vamos entender **quem** está em cada grupo —
olhando só para as características que o K-Means usou.


In [ ]:
# Perfil de cada cluster (sem revelar survived ainda)
print(f"PERFIL DOS CLUSTERS — K={K_escolhido}")
for c in range(K_escolhido):
    sub = df_analise[df_analise["cluster"] == c]
    print(f"\nCluster {c}  (n={len(sub)}, {len(sub)/len(df_analise):.0%} dos passageiros)")
    print(f"  Classe (média):    {sub['pclass'].mean():.2f}  (1=rica, 3=pobre)")
    print(f"  Feminino:          {sub['sex_enc'].mean():.0%}")
    print(f"  Idade média:       {sub['age'].mean():.1f} anos")
    print(f"  Tarifa média:      £{sub['fare'].mean():.2f}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Baseado nos perfis acima, tente nomear cada cluster (ex: "homens jovens de 3ª classe"). Depois faça uma previsão: qual cluster você acha que teve maior taxa de sobrevivência? Registre antes de ir para a Parte 5.</span></div>

*✏️ Nome do Cluster 0: `???`*

*✏️ Nome do Cluster 1: `???`*

*✏️ Nome do Cluster 2: `???`*

*✏️ Previsão (maior para menor sobrevivência): Cluster `???` > `???` > `???`*


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">A Revelação — O que o Algoritmo Descobriu?</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O momento da verdade: os grupos correspondem à sobrevivência?"</p>
    </div>
</div>

O K-Means **nunca viu a coluna `survived`**. Ele agrupou passageiros apenas
pela similaridade das características. Agora vamos revelar se os grupos que
ele criou correspondem aos padrões reais de sobrevivência.


In [ ]:
# ── A GRANDE REVELAÇÃO ────────────────────────────────────────────────────────
print("REVELAÇÃO FINAL — Taxa de Sobrevivência por Cluster")
for c in range(K_escolhido):
    sub = df_analise[df_analise["cluster"] == c]
    taxa = sub["survived"].mean()
    barra = "█" * int(taxa * 30)
    print(f"\nCluster {c}  (n={len(sub):>3})")
    print(f"  Sobrevivência: {taxa:.1%}  {barra}")
    print(f"  {'🟢 Alta' if taxa > 0.5 else ('🟡 Média' if taxa > 0.3 else '🔴 Baixa')}")


In [ ]:
# Comparando visualmente: clusters do K-Means vs sobrevivência real
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("A Revelação — Clusters K-Means vs Sobrevivência Real", fontsize=13, fontweight="bold")

for c in range(K_escolhido):
    mask = labels_final == c
    taxa = df_analise[df_analise["cluster"] == c]["survived"].mean()
    axes[0].scatter(df["age"][mask], df["fare"][mask], color=cmap_k(c/K_escolhido),
                    s=15, alpha=0.6, edgecolors="none", label=f"C{c} — sobrev.: {taxa:.0%}")
axes[0].set_title("Clusters do K-Means", fontweight="bold")
axes[0].set_xlabel("Idade"); axes[0].set_ylabel("Tarifa (£)")
axes[0].legend(fontsize=9)

taxas = [df_analise[df_analise["cluster"]==c]["survived"].mean() for c in range(K_escolhido)]
ns    = [(labels_final==c).sum() for c in range(K_escolhido)]
cores_barra = [cmap_k(c/K_escolhido) for c in range(K_escolhido)]
barras = axes[1].bar([f"Cluster {c}\n(n={ns[c]})" for c in range(K_escolhido)],
                      taxas, color=cores_barra, edgecolor="white", width=0.5)
axes[1].axhline(df["survived"].mean(), color="gray", linestyle="--",
                label=f"Média global ({df['survived'].mean():.0%})")
for b, v in zip(barras, taxas):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.0%}", ha="center", fontweight="bold")
axes[1].set_ylabel("Taxa de Sobrevivência")
axes[1].set_title("Taxa de Sobrevivência por Cluster", fontweight="bold")
axes[1].set_ylim(0, 1.0); axes[1].legend()

plt.tight_layout()
plt.show()


### Avaliação quantitativa

Duas métricas comparam os clusters formados com os rótulos reais — mesmo o
K-Means nunca tendo visto esses rótulos:

- **ARI (Adjusted Rand Index)**: 0 = agrupamento aleatório, 1 = recuperação perfeita
- **NMI (Normalized Mutual Information)**: 0 = clusters não dizem nada sobre o rótulo, 1 = dizem tudo


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(y_real, labels_final)
nmi = normalized_mutual_info_score(y_real, labels_final)

print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi:.4f}")
print()

if ari > 0.3:
    conclusao = "O K-Means recuperou bem os padrões de sobrevivência!"
elif ari > 0.1:
    conclusao = "O K-Means capturou parcialmente os padrões de sobrevivência."
else:
    conclusao = "Os clusters do K-Means são fracos preditores de sobrevivência."

print(f"Conclusão: {conclusao}")
print()
print("O K-Means NUNCA viu 'survived' — agrupou apenas por similaridade.")
print("Se os clusters se alinham com sobrevivência, é porque os mesmos fatores")
print("(gênero, classe, tarifa) que determinam sobrevivência também criam")
print("grupos naturais nos dados.")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 5 (final) — Compare com sua previsão da Missão 4: (a) você acertou a ordem de sobrevivência dos clusters? (b) o valor de ARI/NMI encontrado indica que o K-Means "aprendeu" algo sobre sobrevivência mesmo sem ver o rótulo?</span></div>

*✏️ (a) Acertei minha previsão da Missão 4? `???`*

*✏️ (b) O ARI/NMI mostra que: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 5 (descomente para ver) ───────────────────────────────
# for c in range(K_escolhido):
#     sub = df_analise[df_analise["cluster"] == c]
#     print(f"Cluster {c}: {sub['sex_enc'].mean():.0%} feminino, classe média "
#           f"{sub['pclass'].mean():.1f}, sobrevivência {sub['survived'].mean():.0%}")
# print()
# print("Padrão esperado: o cluster com mais mulheres e melhor classe social")
# print("tende a ter a maior taxa de sobrevivência — reflete o protocolo")
# print("'mulheres e crianças primeiro' e o acesso privilegiado da 1ª classe.")


**✏️ Minha reflexão sobre a aula:**

1. A diferença fundamental entre classificar e fazer clustering é: *...*

2. O K-Means precisa de normalização porque: *...*

3. Eu usaria K-Means em vez de um modelo supervisionado quando: *...*
